# Advanced Problems with Solutions: Python Optimizations — Interning
**Focus:** integer interning, string interning, identity vs equality, compile-time constants, runtime object creation, and safe benchmarking practices.

> Notes:
> - Many results below are **CPython implementation details**, not guarantees of the Python language specification.
> - Avoid writing production logic that depends on `is` for numbers or strings. Use `==` for value comparison.
> - IDs and identity behavior can vary across Python versions, platforms, interpreters, and execution contexts.

## Setup
Run this cell first.

In [1]:
import sys
import timeit
import dis
import platform
from dataclasses import dataclass

print("Python:", sys.version)
print("Implementation:", platform.python_implementation())

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Implementation: CPython


## Problem 1 — Detect the small-integer cache boundary safely

Write a function `same_identity_from_runtime_int(n)` that returns whether two separately-created integer objects with value `n` have the same identity.

Important: avoid accidentally reusing the same compile-time literal object in your test.

Then test values around CPython's usual small-integer cache boundary: `-7` through `258`.

### Solution

In [2]:
def same_identity_from_runtime_int(n: int) -> bool:
    a = int(str(n))
    b = int(str(n))
    return a is b

for n in range(-7, 259):
    if n in {-7, -6, -5, -4, 255, 256, 257, 258}:
        print(f"{n:>4}: {same_identity_from_runtime_int(n)}")

  -7: False
  -6: False
  -5: True
  -4: True
 255: True
 256: True
 257: False
 258: False


### Explanation

On standard CPython, integers from `-5` through `256` are typically pre-allocated and reused.
Values outside that range are usually not reused when constructed dynamically.

This is an optimization detail. Do not depend on it for program correctness.

## Problem 2 — Explain why two large literals may still be identical

Predict the output, then run the code:

```python
a = 10_000
b = 10_000
print(a is b)
```

Then inspect the function's constants with `dis`.

### Solution

In [3]:
def literal_identity_demo():
    a = 10_000
    b = 10_000
    return a is b

print("literal_identity_demo():", literal_identity_demo())

print("\nDisassembly:")
dis.dis(literal_identity_demo)

print("\nConstants:")
print(literal_identity_demo.__code__.co_consts)

literal_identity_demo(): True

Disassembly:
  1           RESUME                   0

  2           LOAD_CONST               1 (10000)
              STORE_FAST               0 (a)

  3           LOAD_CONST               1 (10000)
              STORE_FAST               1 (b)

  4           LOAD_FAST_LOAD_FAST      1 (a, b)
              IS_OP                    0
              RETURN_VALUE

Constants:
(None, 10000)


### Explanation

Even though `10_000` is outside the small-integer cache range, CPython may store the literal once in the function's code object constants.
Both assignments can load the same constant object, so `a is b` may be `True`.

This is not integer interning in the same sense as the small-integer cache. It is compile-time constant reuse.

## Problem 3 — Separate compile-time constant reuse from runtime allocation

Create two functions:

1. `literal_version()` compares two assignments from the same large literal.
2. `runtime_version()` compares two large integers parsed from strings.

Explain why their results may differ.

### Solution

In [4]:
def literal_version():
    x = 50_000
    y = 50_000
    return x is y

def runtime_version():
    x = int("50000")
    y = int("50000")
    return x is y

print("literal_version():", literal_version())
print("runtime_version():", runtime_version())

literal_version(): True
runtime_version(): False


### Explanation

`literal_version()` may return `True` because both variables may load the same constant object from the function's code object.

`runtime_version()` usually returns `False` because each `int("50000")` call creates a new integer object outside the small-integer cache.

The important lesson: identity observations can be affected by **how** objects are created.

## Problem 4 — Fix a bug caused by using `is` instead of `==`

The function below incorrectly uses `is` for value comparison:

```python
def is_status_code_ok(code):
    return code is 200
```

Write tests that reveal why this is unsafe, then fix the function.

### Solution

In [6]:
def bad_is_status_code_ok(code: int) -> bool:
    return code is 200  # intentionally wrong

def good_is_status_code_ok(code: int) -> bool:
    return code == 200

literal_200 = 200
runtime_200 = int("200")

print("bad with literal 200:", bad_is_status_code_ok(literal_200))
print("bad with runtime 200:", bad_is_status_code_ok(runtime_200))
print("good with literal 200:", good_is_status_code_ok(literal_200))
print("good with runtime 200:", good_is_status_code_ok(runtime_200))

assert good_is_status_code_ok(literal_200)
assert good_is_status_code_ok(runtime_200)

bad with literal 200: True
bad with runtime 200: True
good with literal 200: True
good with runtime 200: True


### Explanation

For small integers like `200`, CPython will usually make the bad tests appear to work.
That is exactly why this bug is dangerous: the code can pass tests for cached integers but fail for other values.

For value equality, use `==`.
Use `is` only for identity checks such as `x is None`, singleton sentinels, or intentional object identity comparisons.

## Problem 5 — Build a custom sentinel and compare it correctly

Create a unique sentinel object named `MISSING`.
Write a function `get_or_default(mapping, key, default)` that distinguishes between:

- a missing key
- a key whose value is actually `None`

Use identity correctly.

### Solution

In [7]:
MISSING = object()

def get_or_default(mapping: dict, key, default=None):
    value = mapping.get(key, MISSING)
    if value is MISSING:
        return default
    return value

data = {
    "exists_with_value": 42,
    "exists_with_none": None,
}

print(get_or_default(data, "exists_with_value", "fallback"))
print(get_or_default(data, "exists_with_none", "fallback"))
print(get_or_default(data, "does_not_exist", "fallback"))

assert get_or_default(data, "exists_with_none", "fallback") is None
assert get_or_default(data, "does_not_exist", "fallback") == "fallback"

42
None
fallback


### Explanation

This is a correct use of `is`: we are checking whether a value is exactly the sentinel object `MISSING`.

Unlike strings or integers, this sentinel has no meaningful value equality behavior. Its purpose is unique identity.

## Problem 6 — Investigate string interning rules

Predict which comparisons are likely to be `True`, then run them.

Include:

- identifier-like strings
- strings containing spaces
- strings created at runtime
- manually interned strings with `sys.intern`

### Solution

In [8]:
def string_interning_demo():
    a = "python_identifier"
    b = "python_identifier"

    c = "hello world"
    d = "hello world"

    e = "".join(["python", "_", "identifier"])
    f = "".join(["python", "_", "identifier"])

    g = sys.intern("".join(["hello", " ", "world"]))
    h = sys.intern("".join(["hello", " ", "world"]))

    return {
        "identifier literals": a is b,
        "space-containing literals": c is d,
        "runtime identifier-like strings": e is f,
        "manually interned runtime strings": g is h,
        "value equality still works": e == f == a,
    }

for label, result in string_interning_demo().items():
    print(f"{label:35} {result}")

identifier literals                 True
space-containing literals           True
runtime identifier-like strings     False
manually interned runtime strings   True
value equality still works          True


### Explanation

CPython often interns identifier-like strings, and it may reuse some string literals from a code object.
Runtime-created strings are usually distinct objects unless explicitly interned with `sys.intern`.

Again, use `==` for string values. Use `sys.intern` only when you have a measured reason, such as many repeated dictionary keys or symbol-like strings.

## Problem 7 — Prove that interning can change identity but not equality

Write a function that creates two equal strings dynamically.
Show that they are equal by value, usually different by identity, and then identical after `sys.intern`.

### Solution

In [9]:
def make_dynamic_string():
    return "".join(["data", "_", "pipeline", "_", "stage"])

s1 = make_dynamic_string()
s2 = make_dynamic_string()

print("Before interning:")
print("s1 == s2:", s1 == s2)
print("s1 is s2:", s1 is s2)

s1_i = sys.intern(s1)
s2_i = sys.intern(s2)

print("\nAfter interning:")
print("s1_i == s2_i:", s1_i == s2_i)
print("s1_i is s2_i:", s1_i is s2_i)

Before interning:
s1 == s2: True
s1 is s2: False

After interning:
s1_i == s2_i: True
s1_i is s2_i: True


### Explanation

Interning canonicalizes equal immutable strings so they can share one object.
This can reduce memory usage and speed up some repeated comparisons, but it should be applied intentionally.

## Problem 8 — Benchmark string equality with and without interning

Create two long equal strings dynamically.
Compare the cost of equality checks before and after interning.

Use `timeit` carefully.

### Solution

In [10]:
left = "x" * 10_000 + "END"
right = "".join(["x" * 10_000, "END"])

left_i = sys.intern(left)
right_i = sys.intern(right)

print("left == right:", left == right)
print("left is right:", left is right)
print("left_i is right_i:", left_i is right_i)

non_interned_time = timeit.timeit("left == right", globals=globals(), number=500_000)
interned_time = timeit.timeit("left_i == right_i", globals=globals(), number=500_000)

print(f"non-interned equality time: {non_interned_time:.6f} seconds")
print(f"interned equality time:     {interned_time:.6f} seconds")
print(f"speed ratio:                {non_interned_time / interned_time:.2f}x")

left == right: True
left is right: False
left_i is right_i: True
non-interned equality time: 0.279502 seconds
interned equality time:     0.015086 seconds
speed ratio:                18.53x


### Explanation

For interned strings, CPython can often short-circuit equality because both names reference the same object.
For non-interned but equal long strings, Python may need to compare contents.

Do not overgeneralize from one benchmark. Benchmark your actual workload.

## Problem 9 — Design a memory-conscious symbol table

You receive many repeated category names from an event stream.
Build two lists:

1. raw category strings
2. interned category strings

Then compare how many unique object identities are present.

### Solution

In [11]:
categories = [
    "login", "logout", "purchase", "view_product", "add_to_cart",
    "login", "purchase", "purchase", "view_product", "logout",
] * 10_000

raw_categories = ["".join([c]) for c in categories]
interned_categories = [sys.intern(c) for c in raw_categories]

raw_unique_ids = len({id(c) for c in raw_categories})
interned_unique_ids = len({id(c) for c in interned_categories})
unique_values = len(set(categories))

print("unique values:", unique_values)
print("unique raw object identities:", raw_unique_ids)
print("unique interned object identities:", interned_unique_ids)

assert interned_unique_ids == unique_values

unique values: 5
unique raw object identities: 5
unique interned object identities: 5


### Explanation

Interning is useful when many equal strings are repeated and kept alive.
The interned version stores one canonical object per unique string value.

Trade-off: interned strings may live longer than expected, so avoid interning unbounded or highly unique data such as user-generated IDs.

## Problem 10 — Implement a safe interning cache for arbitrary immutable values

Python automatically interns some integers and strings, but not arbitrary tuples.

Implement `InternPool`:

- `intern(value)` returns a canonical object for equal values.
- It should work for hashable immutable values.
- It should not use `is` to check whether values are equal.
- Demonstrate it with tuples.

### Solution

In [12]:
@dataclass
class InternPool:
    _pool: dict = None

    def __post_init__(self):
        if self._pool is None:
            self._pool = {}

    def intern(self, value):
        try:
            existing = self._pool[value]
        except KeyError:
            self._pool[value] = value
            return value
        return existing

pool = InternPool()

t1 = tuple([1, 2, 3, "alpha"])
t2 = tuple([1, 2, 3, "alpha"])

print("Before custom interning:")
print("t1 == t2:", t1 == t2)
print("t1 is t2:", t1 is t2)

t1_i = pool.intern(t1)
t2_i = pool.intern(t2)

print("\nAfter custom interning:")
print("t1_i == t2_i:", t1_i == t2_i)
print("t1_i is t2_i:", t1_i is t2_i)
print("pool size:", len(pool._pool))

Before custom interning:
t1 == t2: True
t1 is t2: False

After custom interning:
t1_i == t2_i: True
t1_i is t2_i: True
pool size: 1


### Explanation

This is manual canonicalization.
The dictionary uses value equality and hashing to find an existing equal object, then returns the canonical stored object.

This pattern can be useful for symbolic computation, compilers, parsers, AST nodes, and repeated immutable records.

## Problem 11 — Avoid a weak benchmarking trap

The following benchmark is misleading:

```python
timeit.timeit("x is y", setup="x = 500; y = 500")
```

Explain why, then write a better benchmark comparing identity of dynamically-created small and large integers.

### Solution

In [13]:
bad = timeit.timeit("x is y", setup="x = 500; y = 500", number=1_000_000)
print("misleading benchmark:", bad)

better_small = timeit.timeit(
    "int('10') is int('10')",
    number=1_000_000,
)

better_large = timeit.timeit(
    "int('500') is int('500')",
    number=1_000_000,
)

print("dynamic small integer identity checks:", better_small)
print("dynamic large integer identity checks:", better_large)

print("int('10') is int('10'):", int("10") is int("10"))
print("int('500') is int('500'):", int("500") is int("500"))

misleading benchmark: 0.025280699133872986
dynamic small integer identity checks: 0.14321359992027283
dynamic large integer identity checks: 0.17289889976382256
int('10') is int('10'): True
int('500') is int('500'): False


### Explanation

The setup string may compile constants in a way that does not represent runtime object creation.
A better test creates the objects dynamically inside the timed statement.

Even then, this benchmark mostly demonstrates behavior. It is not a reason to use `is` for numeric comparisons.

## Problem 12 — Advanced reasoning: classify each comparison

For each expression, decide whether it is:

- guaranteed by Python semantics
- CPython-specific optimization behavior
- unsafe / should not be used for value logic

Then run the checks.

### Solution

In [15]:
checks = {
    "None is None": None is None,
    "True is True": True is True,
    "int('256') is int('256')": int("256") is int("256"),
    "int('257') is int('257')": int("257") is int("257"),
    "'abc' == ''.join(['a', 'b', 'c'])": "abc" == "".join(["a", "b", "c"]),
    "'abc' is ''.join(['a', 'b', 'c'])": "abc" is "".join(["a", "b", "c"]),
    "sys.intern('abc') is sys.intern(''.join(['a','b','c']))": (
        sys.intern("abc") is sys.intern("".join(["a", "b", "c"]))
    ),
}

for expr, result in checks.items():
    print(f"{expr:65} -> {result}")

None is None                                                      -> True
True is True                                                      -> True
int('256') is int('256')                                          -> True
int('257') is int('257')                                          -> False
'abc' == ''.join(['a', 'b', 'c'])                                 -> True
'abc' is ''.join(['a', 'b', 'c'])                                 -> False
sys.intern('abc') is sys.intern(''.join(['a','b','c']))           -> True


### Classification

| Expression type | Classification | Why |
|---|---|---|
| `None is None` | guaranteed practical singleton behavior | `None` is the singleton null object |
| `True is True` | guaranteed practical singleton behavior | booleans are singleton objects |
| `int('256') is int('256')` | CPython-specific optimization | relies on small-integer cache |
| `int('257') is int('257')` | unsafe for value logic | usually false, but identity is irrelevant |
| string `==` | correct value comparison | compares contents |
| runtime string `is` | unsafe for value logic | object identity may vary |
| `sys.intern(...) is sys.intern(...)` | intentional identity canonicalization | manual interning returns canonical string |

## Final Takeaways

1. `==` asks: **Do these objects have the same value?**
2. `is` asks: **Are these names bound to the exact same object?**
3. CPython caches small integers, commonly `-5` through `256`.
4. CPython may reuse constants inside code objects.
5. Some strings are automatically interned; strings can also be manually interned with `sys.intern`.
6. Interning is an optimization technique, not a substitute for correct value comparison.
7. Use `is` for `None`, singleton sentinels, and intentional identity checks.